In [ ]:
!pip install datasets
!pip install sentence-transformers
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 76.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import faiss
import os

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

In [ ]:
import pandas as pd

column_names = [
    "target",
    "title",
    "text"
]

train_df = pd.read_csv(
    "train.txt",
    sep="\t",
    names=column_names,
    skiprows=1
)

train_df.head()

,target,title,text
0,BACKGROUND,The emergence of HIV as a chronic condition me...,NaN
1,BACKGROUND,This paper describes the design and evaluation...,NaN
2,METHODS,This study is designed as a randomised control...,NaN
3,METHODS,The intervention group will participate in the...,NaN
4,METHODS,The program is based on self-efficacy theory a...,NaN


In [ ]:
print(train_df.shape)
train_df.columns

(598988, 3)


Index(['target', 'title', 'text'], dtype='object')

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 598988 entries, 0 to 598987
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   target  598988 non-null  object 
 1   title   551507 non-null  object 
 2   text    0 non-null       float64
dtypes: float64(1), object(2)
memory usage: 13.7+ MB


In [ ]:
train_df.isnull().sum()

,0
target,0
title,47481
text,598988


In [ ]:
train_df = train_df.drop(columns=["text"])

train_df = train_df.dropna()

train_df.reset_index(drop=True, inplace=True)

print(train_df.shape)

train_df.head()

(551507, 2)


,target,title
0,BACKGROUND,The emergence of HIV as a chronic condition me...
1,BACKGROUND,This paper describes the design and evaluation...
2,METHODS,This study is designed as a randomised control...
3,METHODS,The intervention group will participate in the...
4,METHODS,The program is based on self-efficacy theory a...


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model Loaded Successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully!


In [ ]:
train_df = train_df.head(5000)
print(train_df.shape)

(5000, 2)


In [ ]:
embeddings = model.encode(
    train_df["title"].tolist(),
    show_progress_bar=True
)

print("Embeddings Shape:", embeddings.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings Shape: (5000, 384)


In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])

index.add(embeddings)

print("FAISS Index Created Successfully!")

FAISS Index Created Successfully!


In [ ]:
def search_papers(query, k=5):

    query_embedding = model.encode([query]).astype("float32")

    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, k)

    print("="*60)
    print("Search Query:", query)
    print("="*60)

    for i, idx in enumerate(indices[0]):
        print(f"\nResult {i+1}")
        print("Category :", train_df.iloc[idx]["target"])
        print("Paper :", train_df.iloc[idx]["title"])

In [ ]:
search_papers("deep learning for brain tumor detection")

Search Query: deep learning for brain tumor detection

Result 1
Category : CONCLUSIONS
Paper : An approach to lung cancer staging with PET-CT improves discrimination between N0-1 and N2-3 .

Result 2
Category : CONCLUSIONS
Paper : Our results are the basis for dose-escalation studies to improve local tumour control .

Result 3
Category : METHODS
Paper : Intermediate-risk patients were all other patients with local or regional tumors .

Result 4
Category : RESULTS
Paper : A total of 189 patients were recruited , 98 in the PET-CT group and 91 in the CWU group .

Result 5
Category : RESULTS
Paper : Cox regression models that took TNM staging or the residual tumor classification and tumor site into account also found significant differences at 10 years .
